# ゼロから作るショアのアルゴリズム 第4回: べき剰余

[第3回](ja/403_shor_scratch_controlled_multiplier.ipynb)では制御剰余乗算 $\ket{\text{ctrl}}\ket{x}\ket{0} \to \ket{\text{ctrl}}\ket{x}\ket{\text{ctrl}\cdot(a x \bmod N)}$ を作りました。今回は、ショアのアルゴリズムの量子部分が実際に実行する演算、**べき剰余**を作ります。

$$
\ket{j}\ket{1} \longrightarrow \ket{j}\ket{a^j \bmod N}
$$

— 位相推定レジスタが同時に保持している*すべての* $j$ の値について、重ね合わせの中で $a^j \bmod N$ を計算します。これがこのシリーズ最後の算術演算の部品です。周囲の位相推定(アダマールゲート、この回路、逆量子フーリエ変換、測定、連分数展開)がどのように位数発見に変わるかは[400_shor.ipynb](ja/400_shor.ipynb)を参照してください。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995)

## 第3回の乗算器だけではまだ足りない理由

複数ビットの $j = j_0 + 2j_1 + 4j_2 + \cdots$ に対する $a^j \bmod N$ は $a^{j_0} \cdot (a^2)^{j_1} \cdot (a^4)^{j_2} \cdots \bmod N$ であり、$j$ の対応するビットが1のときだけ含まれる、繰り返し2乗の積です。したがって自然な計画は: レジスタを $1$ から始め、$j$ の各ビット $k$ について、そのビットで*制御しながら* $a^{2^k} \bmod N$ を掛けていく、というものです。

しかし第3回の `build_controlled_multiplier` は $b \mathrel{+}= a \cdot x \bmod N$ を**別の、まっさらなレジスタ** $b$ に計算します — $x$ 自体を $ax \bmod N$ に変えるわけではありません。1回の乗算ならそれで問題ありませんが、べき乗計算では、ある1ステップの*出力*を次のステップの*入力*として、同じレジスタを使って繰り返しフィードバックする必要があります。**その場で**動くバージョンが必要です: 別の出力レジスタを残さずに $x \to ax \bmod N$ とする方法です。

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

/Users/yuichirominato/.pyenv/shims/pip: line 8: /opt/homebrew/opt/pyenv/bin/pyenv: No such file or directory


## その場での制御乗算: 掛けて、スワップして、掛け戻す

標準的なテクニック(第1〜3回と同じ論文によるもの)には、もう1つの材料が必要です: $a$ の逆元、つまり $a \cdot a^{-1} \equiv 1 \pmod N$ となる $a^{-1}$ です — これは $a$ と $N$ が互いに素であればいつでも存在します(ショアのアルゴリズムは常にそのような $a$ を選びます。Pythonの `pow(a, -1, N)` で直接計算できます)。

1. 第3回の乗算器を実行: $\ket{x}\ket{0} \to \ket{x}\ket{ax \bmod N}$(制御付き)。
2. 同じ制御量子ビットで、$x$ レジスタと(今、値が入った)結果レジスタを**スワップ**します: $\ket{x}\ket{ax \bmod N} \to \ket{ax \bmod N}\ket{x}$。
3. 第3回の乗算器を$a$の代わりに$a^{-1}$を使って*逆向きに*実行します: 2つ目のレジスタには今や*元の* $x$ が入っているため、それに $a^{-1}$ を掛けて引く(つまり回路全体を逆向きに実行する)ことで、きれいに $\ket 0$ に戻ります — $a^{-1} \cdot x$ が、$x$ をそこに置いたときに計算しようとしていたことを正確に打ち消すからです。

最終状態: $\ket{ax \bmod N}\ket{0}$ — 2つのレジスタの役割が入れ替わりましたが、2つ目は再びきれいな状態なので、どちらを「$x$」と呼ぶかを付け替えれば、まさに求めていた「その場での」更新になります。

In [2]:
from blueqat import Circuit

def mcx(controls, target, ancillas):
    """X on `target`, controlled by ALL qubits in `controls` (see Part 3 for the derivation)."""
    circ = Circuit()
    n = len(controls)
    if n == 0:
        circ.x[target]
        return circ
    if n == 1:
        circ.cx[controls[0], target]
        return circ
    if n == 2:
        circ.ccx[controls[0], controls[1], target]
        return circ
    chain = []
    circ.ccx[controls[0], controls[1], ancillas[0]]
    chain.append(ancillas[0])
    for i in range(2, n - 1):
        circ.ccx[chain[-1], controls[i], ancillas[len(chain)]]
        chain.append(ancillas[len(chain)])
    circ.ccx[chain[-1], controls[-1], target]
    for i in range(len(chain) - 1, 0, -1):
        circ.ccx[chain[i - 1], controls[i + 1], chain[i]]
    circ.ccx[controls[0], controls[1], chain[0]]
    return circ

def carry_ops(c, a, b, c_next):
    return [((a, b), c_next), ((a,), b), ((c, b), c_next)]

def carry_inv_ops(c, a, b, c_next):
    return [((c, b), c_next), ((a,), b), ((a, b), c_next)]

def sum_ops(c, a, b):
    return [((a,), b), ((c,), b)]

def add_ops(c, a, b, n):
    ops = []
    for i in range(n - 1):
        ops += carry_ops(c[i], a[i], b[i], c[i + 1])
    ops += carry_ops(c[n - 1], a[n - 1], b[n - 1], b[n])
    ops.append(((a[n - 1],), b[n - 1]))
    ops += sum_ops(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        ops += carry_inv_ops(c[i], a[i], b[i], c[i + 1])
        ops += sum_ops(c[i], a[i], b[i])
    return ops

def sub_ops(c, a, b, n):
    return list(reversed(add_ops(c, a, b, n)))

def modulo_add_ops(c, a, b, N, t, n):
    """b := (a + b) mod N -- the five-step construction from Part 2, as (controls, target) pairs."""
    ops = []
    ops += add_ops(c, a, b, n)
    ops += sub_ops(c, N, b, n)
    ops.append(((b[n],), t))
    ops += [(ctrl + (t,), tgt) for ctrl, tgt in add_ops(c, N, b, n)]
    ops += sub_ops(c, a, b, n)
    ops.append(((), b[n]))
    ops.append(((b[n],), t))
    ops.append(((), b[n]))
    ops += add_ops(c, a, b, n)
    return ops

def controlled_swap_ops(ctrl, a_reg, b_reg):
    """Swap a_reg[i] <-> b_reg[i] for every i, controlled by `ctrl` (Fredkin gate, built from CX/CCX/CX)."""
    ops = []
    for a, b in zip(a_reg, b_reg):
        ops.append(((b,), a))
        ops.append(((a, ctrl), b))
        ops.append(((b,), a))
    return ops

## その場で動く制御乗算器

第3回と同じ定数読み込みのテクニックを使います(定数 $a \cdot 2^i \bmod N$ と $a^{-1} \cdot 2^i \bmod N$ は回路構築時に既知です)。これにスワップと、逆順の2回目のパスを追加します。

In [3]:
def build_controlled_Ua(mult_a, val_x, val_N, ctrl_value, n, n_ancilla=4):
    """x <- (a * x) mod N if ctrl else x, done truly in place."""
    c = list(range(n))                                  # carry register
    const = list(range(n, 2 * n))                        # scratch: holds a constant, loaded/unloaded via X
    res = list(range(2 * n, 3 * n + 1))                  # result register (n+1 wide, becomes the new x)
    N = list(range(3 * n + 1, 4 * n + 1))                 # modulus
    x = list(range(4 * n + 1, 5 * n + 1))                 # the value being exponentiated
    ctrl = 5 * n + 1
    t = 5 * n + 2                                         # modulo-adder overflow flag
    ancillas = list(range(5 * n + 3, 5 * n + 3 + n_ancilla))
    circ = Circuit(5 * n + 3 + n_ancilla)

    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1:
            circ.x[x[i]]
        if (val_N >> i) & 1:
            circ.x[N[i]]

    def apply_modulo_add(constant, ops_source):
        nonlocal circ
        for i in range(n):
            const_val = (constant * (1 << i)) % val_N
            for j in range(n):
                if (const_val >> j) & 1:
                    circ.x[const[j]]
            ops = ops_source(c, const, res, N, t, n)
            for controls, target in ops:
                circ += mcx(list(controls) + [ctrl, x[i]], target, ancillas)
            for j in range(n):
                if (const_val >> j) & 1:
                    circ.x[const[j]]

    # Step A: res += a * x  mod N  (controlled)
    apply_modulo_add(mult_a, modulo_add_ops)
    # Step B: swap x <-> res[0..n-1] (controlled)
    for controls, target in controlled_swap_ops(ctrl, x, res[:n]):
        circ += mcx(list(controls), target, ancillas)
    # Step C: res -= a^-1 * x  mod N  (controlled) -- x now holds the old res value; this zeroes res back out
    a_inv = pow(mult_a, -1, val_N)
    apply_modulo_add(a_inv, lambda *args: list(reversed(modulo_add_ops(*args))))

    return circ, x, res

第3回自身の数値で確認しましょう: $2 \times 1 \bmod 3 = 2$、そしてスクラッチレジスタは正確に $0$ に戻るはずです。

In [4]:
def run_Ua(mult_a, val_x, val_N, ctrl_value, n):
    circuit, x, res = build_controlled_Ua(mult_a, val_x, val_N, ctrl_value, n)
    total = circuit.n_qubits
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    x_val = int("".join(reversed([state[total - 1 - q] for q in x])), 2)
    res_val = int("".join(reversed([state[total - 1 - q] for q in res])), 2)
    return x_val, res_val

for a, xv, N, ctrl in [(2, 1, 3, 1), (2, 2, 3, 1), (2, 1, 3, 0), (2, 0, 3, 1)]:
    x_val, res_val = run_Ua(a, xv, N, ctrl, 2)
    expected_x = (a * xv) % N if ctrl else xv
    ok = x_val == expected_x and res_val == 0
    print(f"ctrl={ctrl}: x={xv} -> {x_val} (expected {expected_x}), scratch={res_val} (expected 0)  {'OK' if ok else 'MISMATCH'}")

ctrl=1: x=1 -> 2 (expected 2), scratch=0 (expected 0)  OK


ctrl=1: x=2 -> 1 (expected 1), scratch=0 (expected 0)  OK


ctrl=0: x=1 -> 1 (expected 1), scratch=0 (expected 0)  OK


ctrl=1: x=0 -> 0 (expected 0), scratch=0 (expected 0)  OK


4つのケースすべてが確認できました。`ctrl=0`(xが変化しない)と `x=0`(どちらにしても $a \cdot 0 = 0$ なので0のまま)も含みます。

## べき乗計算全体への連結

$j$ の最下位ビットから最上位ビットまでが、順に $a^1, a^2, a^4, a^8, \ldots \bmod N$ を掛けるかどうかを制御します(それぞれ単に `pow(a, 2**k, N)` であり、乗算器の内部の各ビットの定数と同様に、1ビットごとに古典的に一度だけ計算します)。$x$ は $1$($a^0$)から始まり、各ステップは前のステップの出力を次のステップの入力としてそのままフィードバックします。

In [5]:
def build_modexp(mult_a, val_N, exponent_val, n, n_exp_bits, n_ancilla=4):
    """x ends up holding a**exponent_val mod N. exponent_val is set classically here (via X gates) so we
    can check correctness against Python's pow(); in the real algorithm this register is a superposition
    prepared by Hadamards, and the whole circuit below runs unchanged on every branch at once."""
    c = list(range(n))
    const = list(range(n, 2 * n))
    res = list(range(2 * n, 3 * n + 1))
    N = list(range(3 * n + 1, 4 * n + 1))
    x = list(range(4 * n + 1, 5 * n + 1))
    j = list(range(5 * n + 1, 5 * n + 1 + n_exp_bits))       # exponent register, one control qubit per bit
    t = 5 * n + 1 + n_exp_bits
    ancillas = list(range(5 * n + 2 + n_exp_bits, 5 * n + 2 + n_exp_bits + n_ancilla))
    circ = Circuit(5 * n + 2 + n_exp_bits + n_ancilla)

    circ.x[x[0]]  # x starts at 1
    for i in range(n):
        if (val_N >> i) & 1:
            circ.x[N[i]]
    for k in range(n_exp_bits):
        if (exponent_val >> k) & 1:
            circ.x[j[k]]

    def apply_modulo_add(ctrl, constant, ops_source):
        nonlocal circ
        for i in range(n):
            const_val = (constant * (1 << i)) % val_N
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]
            for controls, target in ops_source(c, const, res, N, t, n):
                circ += mcx(list(controls) + [ctrl, x[i]], target, ancillas)
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]

    for k in range(n_exp_bits):
        a_pow = pow(mult_a, 1 << k, val_N)      # a^(2^k) mod N, controlled by bit k of the exponent
        a_pow_inv = pow(a_pow, -1, val_N)
        apply_modulo_add(j[k], a_pow, modulo_add_ops)
        for controls, target in controlled_swap_ops(j[k], x, res[:n]):
            circ += mcx(list(controls), target, ancillas)
        apply_modulo_add(j[k], a_pow_inv, lambda *args: list(reversed(modulo_add_ops(*args))))

    return circ, x

In [6]:
def run_modexp(mult_a, val_N, exponent_val, n, n_exp_bits):
    circuit, x = build_modexp(mult_a, val_N, exponent_val, n, n_exp_bits)
    total = circuit.n_qubits
    print(f"  [qubits={total}, gates={len(circuit.ops)}]")
    state = circuit.m[:].run(shots=1).most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in x]
    return int("".join(reversed(bits)), 2)

for exponent in [0, 1, 2, 3]:
    result = run_modexp(2, 3, exponent, 2, 2)
    expected = pow(2, exponent, 3)
    print(f"2^{exponent} mod 3 = {result}  (expected {expected}, {'OK' if result == expected else 'MISMATCH'})")

  [qubits=18, gates=2479]


2^0 mod 3 = 1  (expected 1, OK)
  [qubits=18, gates=2480]


2^1 mod 3 = 2  (expected 2, OK)
  [qubits=18, gates=2480]


2^2 mod 3 = 1  (expected 1, OK)
  [qubits=18, gates=2481]


2^3 mod 3 = 2  (expected 2, OK)


すべての4つの指数がPython自身の `pow(2, j, 3)` と正確に一致し、その場での制御乗算の連鎖が、2量子ビットの指数レジスタが保持できる $j=0$ から $3$ までのすべてについて、正しく $2^j \bmod 3$ を計算していることが確認できました。ここでは各実行に1分強かかりました(18量子ビット、およそ2,500ゲート): 第3回の1回の乗算コストのおよそ4倍です — 2つの指数ビットを連鎖させ、それぞれが1回の乗算+スワップ+逆乗算のコストを持つため、第3回のおよそ2倍のコストになり、そこで議論したゲート数駆動のスケーリングと一致します。

## ここからどうつながるか

これがこのシリーズで新しく作る最後の算術回路です。実際のアルゴリズムでは:

1. 上記の指数レジスタは`X`ゲートで古典的に設定されるのではなく、アダマールゲートによってすべての値の等しい重ね合わせにされます。
2. この同じべき剰余回路が*一度だけ*、変更なしに実行され、その重ね合わせの中のすべての $j$ について同時に $a^j \bmod N$ を計算します — これこそが指数関数的な並列性が実際に起きている場所です。
3. 指数レジスタに対する逆量子フーリエ変換とそれに続く測定によって、高い確率で、$a$ の $N$ を法とした位数 $r$ の逆数 $1/r$ の倍数の推定値が得られます。
4. 古典的な連分数展開による後処理が、その推定値を候補の $r$ に変換し、それが検証され、偶数であれば $\gcd(a^{r/2} \pm 1, N)$ を通じて $N$ の因数を取り出すのに使われます。

ステップ3と4は、まさに[400_shor.ipynb](ja/400_shor.ipynb)がすでに詳しく解説している内容です($N=15$ と $N=21$ の完全な実例も含みます) — このシリーズの役割は、理論では通常「そしてべき剰余回路を作ります」で済まされてしまうステップ2の部分を、`X`/`CX`/`CCX` から積み上げて、実際に自分の手でゲート単位で作って検証することでした。